In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

base = "/kaggle/input"
print(os.listdir(base))


In [ ]:
os.listdir("/kaggle/input/biomass-data")


In [ ]:
!cp /kaggle/input/biomass-data/*.py /kaggle/working/
!cp /kaggle/input/biomass-data/train_split.csv /kaggle/working/
!cp /kaggle/input/biomass-data/val_split.csv /kaggle/working/


In [ ]:
import os
os.listdir("/kaggle/working")


In [ ]:
%%writefile /kaggle/working/train.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from model import BiomassModel
from dataset import BiomassDataset
import torchvision.transforms as transforms
from tqdm import tqdm

# Paths inside Kaggle
train_csv = "/kaggle/working/train_split.csv"
val_csv = "/kaggle/working/val_split.csv"
root_dir = "/kaggle/working"   # folder containing train/

batch_size = 16
epochs = 5
lr = 1e-4

# Image transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Dataset & loaders
train_dataset = BiomassDataset(train_csv, root_dir, transform)
val_dataset = BiomassDataset(val_csv, root_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Model
model = BiomassModel(n_outputs=5).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch+1}/{epochs}")
    for imgs, targets in tqdm(train_loader, desc="Training"):
        imgs = imgs.to(device)
        targets = targets.float().to(device)   # shape: [B, 5]

        preds = model(imgs)                    # shape: [B, 5]
        loss = criterion(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Train Loss: {total_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, targets in tqdm(val_loader, desc="Validating"):
            imgs = imgs.to(device)
            targets = targets.float().to(device)

            preds = model(imgs)
            loss = criterion(preds, targets)
            val_loss += loss.item()

    print(f"Val Loss: {val_loss/len(val_loader):.4f}")

# Save model
torch.save(model.state_dict(), "/kaggle/working/biomass_model.pth")
print("Training complete. Model saved to biomass_model.pth")


In [ ]:
%%writefile /kaggle/working/model.py
import torch
import torch.nn as nn
from torchvision import models

class BiomassModel(nn.Module):
    def __init__(self, n_outputs=5):
        super().__init__()

        # Load EfficientNet-B0 WITHOUT pretrained weights
        self.backbone = models.efficientnet_b0(weights=None)

        # Replace classifier head with regression layer
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_outputs)
        )

    def forward(self, x):
        return self.backbone(x)


In [ ]:
!ls /kaggle/input/image-data



In [ ]:
!cp -r /kaggle/input/image-data/train /kaggle/working/
!cp -r /kaggle/input/image-data/test /kaggle/working/


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working", topdown=True):
    level = root.replace("/kaggle/working", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")


In [ ]:
%%writefile /kaggle/working/dataset.py
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class BiomassDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # CSV gives: train/IDxxxx.jpg or test/IDxxxx.jpg
        relative_path = row["image_path"]

        # Correct full path
        img_path = os.path.join(self.root_dir, relative_path)

        # Load image
        image = Image.open(img_path).convert("RGB")

        # Target value
        target = float(row["target"])

        if self.transform:
            image = self.transform(image)

        return image, target

   


In [ ]:
%%writefile /kaggle/working/train.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from model import BiomassModel
from dataset import BiomassDataset
import torchvision.transforms as transforms
from tqdm import tqdm

# CSV paths
train_csv = "/kaggle/working/train_split.csv"
val_csv   = "/kaggle/working/val_split.csv"

root_dir = "/kaggle/working" 

batch_size = 16
epochs = 5
lr = 1e-4

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Dataset & loaders
train_dataset = BiomassDataset(train_csv, root_dir, transform)
val_dataset   = BiomassDataset(val_csv, root_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# MODEL → single output (1 target column)
model = BiomassModel(n_outputs=1).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Training Loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch+1}/{epochs}")

    for imgs, targets in tqdm(train_loader, desc="Training"):
        imgs = imgs.to(device)
        targets = targets.float().to(device)        # shape: [B]

        preds = model(imgs).squeeze()               # shape: [B]

        loss = criterion(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Train Loss: {total_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, targets in tqdm(val_loader, desc="Validating"):
            imgs = imgs.to(device)
            targets = targets.float().to(device)

            preds = model(imgs).squeeze()

            loss = criterion(preds, targets)
            val_loss += loss.item()

    print(f"Val Loss: {val_loss/len(val_loader):.4f}")

# Save model
torch.save(model.state_dict(), "/kaggle/working/biomass_model.pth")
print("Training complete. Model saved.")


In [ ]:
import os
import shutil

base = "/kaggle/working/train"
nested = os.path.join(base, "train")

# Check if nested folder exists
if os.path.exists(nested):
    print("Flattening folder structure...")

    # Move all files from train/train/ → train/
    for file in os.listdir(nested):
        src = os.path.join(nested, file)
        dst = os.path.join(base, file)
        shutil.move(src, dst)

    # Delete the now-empty train/train folder
    shutil.rmtree(nested)
    
    print("Flattened successfully!")
else:
    print("No nested folder found.")


In [ ]:
import os
import shutil

# Paths
test_root = "/kaggle/working/test"
nested_test = os.path.join(test_root, "test")

# If nested folder exists, move everything up
if os.path.exists(nested_test):
    for filename in os.listdir(nested_test):
        src = os.path.join(nested_test, filename)
        dst = os.path.join(test_root, filename)

        if os.path.isfile(src):
            shutil.move(src, dst)

    # Remove the empty nested folder
    shutil.rmtree(nested_test)
    print("Nested test folder removed and files moved up.")
else:
    print("No nested test folder found.")


In [ ]:
print(os.path.exists("/kaggle/working/test/test"))


In [ ]:
ds = BiomassDataset("train_split.csv", "/kaggle/working")
print(len(ds))
img, tgt = ds[0]
print(type(img), tgt)


In [ ]:
%%writefile /kaggle/working/model.py
import torch
import torch.nn as nn
from torchvision import models

class BiomassModel(nn.Module):
    def __init__(self, n_outputs=1):      # <<< change to 1
        super().__init__()

        # Load EfficientNet-B0 WITHOUT pretrained weights
        self.backbone = models.efficientnet_b0(weights=None)

        # Replace classifier head with regression layer
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_outputs)      # <<< now outputs 1 value
        )

    def forward(self, x):
        return self.backbone(x).squeeze(1)  # <<< makes output [batch]



In [ ]:
import model
print(model.__file__)


In [ ]:
!python /kaggle/working/train.py


In [ ]:
import os
os.listdir()
